# S01 — Fit3D shoulder exercises: truth, live angle and streaming features (PC)

For every subject × camera × shoulder exercise:

1. MediaPipe Pose (image + world landmarks), cached in `data/fit3d/cache` — the same code as notebook 01;
2. finds which of the 25 mocap joints are the **shoulders, elbows and hips** from the data itself;
3. **truth** = 3D shoulder elevation: trunk axis (mid-shoulder → mid-hip) vs upper arm (shoulder → elbow);
4. labels the camera *front / half / profile* from the shoulder line;
5. builds the 15 shoulder features **frame by frame** with `shoulder_features.stream_shoulder_features`
   (exactly what the live system will compute: causal filter, session-long statistics, torso scale);
6. gates: truth ROM ≥ 40°, angle coverage ≥ 50 %, alignment r ≥ 0.7;
7. saves `data/fit3d/feat_fit3d_shoulder.npz` and prints the **raw baseline** per view.

MediaPipe on ~190 new videos takes about 1–2 hours on a laptop. Stopped half way? Run the cell again:
finished videos are cached.

**Licence:** Fit3D is non-commercial, single user, no redistribution. Everything written here stays in
`data/fit3d/` (git-ignored). Never push it or share it.

In [2]:
# ---------------- settings: edit these ----------------
import os, sys, glob, json, time
REPO       = r'F:\namal sir project\github repo\AI-based-Physiotherapy-project-'   # <-- your repo folder
FIT3D_ROOT = r'F:\namal sir project\fit3d dataset\sholders\train'                  # <-- folder holding s03, s04, ...
OUT        = os.path.join(REPO, 'data', 'fit3d')                                   # git-ignored
WANTED     = ['side_lateral_raise', 'dumbbell_scaptions', 'dumbbell_overhead_shoulder_press',
              'neutral_overhead_shoulder_press', 'overhead_trap_raises', 'w_raise']
EXTRACT_ONLY = False
STEP       = 2          # 50 fps -> 25 fps (close to a 30 fps webcam)
N_WORKERS  = max(1, (os.cpu_count() or 2) - 1)
THRESHOLD, MAX_LAG, MIN_ROM, MIN_COV = 0.70, 30, 40.0, 50.0

os.makedirs(os.path.join(OUT, 'cache'), exist_ok=True)
sys.path[:0] = [REPO, os.path.join(REPO, 'pose'), os.path.join(REPO, 'scripts')]
subjects = sorted(os.path.basename(p) for p in glob.glob(os.path.join(FIT3D_ROOT, 's*')))
EXERCISES = [e for e in WANTED
             if glob.glob(os.path.join(FIT3D_ROOT, 's*', 'camera_parameters', '*', e + '.json'))]
print('subjects :', subjects)
print('exercises:', EXERCISES)
missing_ex = sorted(set(WANTED) - set(EXERCISES))
if missing_ex:
    print('not downloaded (skipped):', missing_ex)
assert subjects and EXERCISES, 'FIT3D_ROOT must be the folder that holds s03, s04, ...'

subjects : ['s03', 's04', 's05', 's07', 's08', 's09', 's10', 's11']
exercises: ['side_lateral_raise', 'dumbbell_scaptions', 'dumbbell_overhead_shoulder_press', 'neutral_overhead_shoulder_press', 'overhead_trap_raises', 'w_raise']


In [3]:
import numpy as np, pandas as pd
import angles as angles_2d
import benchmark_common as bc
from features import overlap_rows
from shoulder_features import SH_FEATURE_NAMES, IDX, stream_shoulder_features, truth_elevation

## 1 — MediaPipe on every video (cached)

In [4]:
from concurrent.futures import ProcessPoolExecutor
import multiprocessing as mproc

def cache_path(subj, cam, ex):
    return os.path.join(OUT, 'cache', f'{subj}_{cam}_{ex}_s{STEP}_w.npz')

def extract_one(job):
    video, cache = job
    if os.path.exists(cache):
        return cache, 'cached', 0.0
    import cv2
    from landmarks import LandmarkExtractor
    t0 = time.time()
    cap = cv2.VideoCapture(video)
    fps = cap.get(cv2.CAP_PROP_FPS) or 50.0
    W, H = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    ext = LandmarkExtractor(model_complexity=1)
    lms, wds, frames, i = [], [], [], 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if i % STEP == 0:
            n, w = ext.process_with_world(frame)
            lms.append(n); wds.append(w); frames.append(i)
        i += 1
    cap.release(); ext.close()
    np.savez_compressed(cache, lm=np.asarray(lms, np.float32), world=np.asarray(wds, np.float32),
                        frames=np.asarray(frames), size=np.asarray([W, H]), fps=fps / STEP)
    return cache, 'new', time.time() - t0

# Recordings are found from the camera files, so the videos are only needed for
# recordings that are not cached yet (you can run MediaPipe on your laptop and
# copy only data/fit3d/cache/ to the server).
jobs, index, missing = [], [], []
for sd in sorted(glob.glob(os.path.join(FIT3D_ROOT, 's*'))):
    subj = os.path.basename(sd)
    for ex in EXERCISES:
        for cp in sorted(glob.glob(os.path.join(sd, 'camera_parameters', '*', ex + '.json'))):
            cam = os.path.basename(os.path.dirname(cp))
            v, c = os.path.join(sd, 'videos', cam, ex + '.mp4'), cache_path(subj, cam, ex)
            if os.path.exists(c) or os.path.exists(v):
                jobs.append((v, c)); index.append((subj, cam, ex))
            else:
                missing.append(f'{subj}/{cam}/{ex}')
print(len(jobs), 'recordings;', sum(os.path.exists(c) for _, c in jobs), 'already cached;',
      len(missing), 'with neither video nor cache (skipped)')

todo = [j for j in jobs if not os.path.exists(j[1])]
t0 = time.time()
if todo and 'fork' in mproc.get_all_start_methods() and N_WORKERS > 1:      # Linux / ADA
    with ProcessPoolExecutor(N_WORKERS, mp_context=mproc.get_context('fork')) as pool:
        for k, (c, how, sec) in enumerate(pool.map(extract_one, todo), 1):
            print(f'[{k}/{len(todo)}] {os.path.basename(c)}  {sec:.0f}s', flush=True)
else:                                     # Windows: no fork in notebooks -> one by one
    for k, job in enumerate(todo, 1):
        c, how, sec = extract_one(job)
        print(f'[{k}/{len(todo)}] {os.path.basename(c)}  {sec:.0f}s', flush=True)
# stopped half way? just run this cell again - finished videos are cached and skipped
print(f'done in {(time.time() - t0) / 60:.1f} min')
if EXTRACT_ONLY:
    raise SystemExit('EXTRACT_ONLY: cache written to ' + os.path.join(OUT, 'cache') + ' - copy that folder to the server')

172 recordings; 0 already cached; 0 with neither video nor cache (skipped)


f:\namal sir project\github repo\AI-based-Physiotherapy-project-\.venv\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


[1/172] s03_50591643_side_lateral_raise_s2_w.npz  21s
[2/172] s03_58860488_side_lateral_raise_s2_w.npz  21s
[3/172] s03_60457274_side_lateral_raise_s2_w.npz  17s
[4/172] s03_65906101_side_lateral_raise_s2_w.npz  20s
[5/172] s03_50591643_dumbbell_scaptions_s2_w.npz  28s
[6/172] s03_58860488_dumbbell_scaptions_s2_w.npz  32s
[7/172] s03_60457274_dumbbell_scaptions_s2_w.npz  28s
[8/172] s03_65906101_dumbbell_scaptions_s2_w.npz  29s
[9/172] s03_50591643_dumbbell_overhead_shoulder_press_s2_w.npz  34s
[10/172] s03_58860488_dumbbell_overhead_shoulder_press_s2_w.npz  26s
[11/172] s03_60457274_dumbbell_overhead_shoulder_press_s2_w.npz  23s
[12/172] s03_65906101_dumbbell_overhead_shoulder_press_s2_w.npz  28s
[13/172] s03_50591643_neutral_overhead_shoulder_press_s2_w.npz  35s
[14/172] s03_58860488_neutral_overhead_shoulder_press_s2_w.npz  32s
[15/172] s03_60457274_neutral_overhead_shoulder_press_s2_w.npz  31s
[16/172] s03_65906101_neutral_overhead_shoulder_press_s2_w.npz  27s
[17/172] s03_50591643

## 2 — Which Fit3D joints are the shoulders, elbows and hips?

Every recording projects all 25 mocap joints into its own camera and picks, for each MediaPipe
landmark (shoulder 11/12, elbow 13/14, hip 23/24), the joint that lands on it. The majority answer
over all recordings is used everywhere.

In [5]:
MP = {'left': (11, 13, 23), 'right': (12, 14, 24)}   # shoulder, elbow, hip

def load_json(p):
    with open(p, encoding='utf-8') as f:
        return json.load(f)

def load_joints(p):
    d = load_json(p)
    a = np.asarray(d.get('joints3d_25', d) if isinstance(d, dict) else d, float)
    assert a.ndim == 3 and a.shape[1:] == (25, 3), a.shape
    return a

def load_camera(p):
    d = load_json(p)
    ex = d['extrinsics']
    it = d.get('intrinsics_wo_distortion') or d.get('intrinsics_w_distortion') or d.get('intrinsics')
    return (np.asarray(ex['R'], float).reshape(3, 3), np.asarray(ex['T'], float).reshape(3),
            np.asarray(it['f'], float).reshape(-1), np.asarray(it['c'], float).reshape(-1))

def to_camera(X, R, T, mode):
    return [(X - T) @ R.T, X @ R.T + T, (X - T) @ R, X @ R + T][mode]

def project(Xc, f, c):
    with np.errstate(invalid='ignore', divide='ignore'):
        return Xc[..., :2] / Xc[..., 2:3] * f + c

def match(J, lm_px, R, T, f, c):
    best = None
    for mode in range(4):
        Xc = to_camera(J, R, T, mode)
        if np.nanmedian(Xc[..., 2]) <= 0:
            continue
        P = project(Xc, f, c)
        mapping, errs = {}, []
        for side, idx in MP.items():
            pick = []
            for m in idx:
                med = np.nanmedian(np.linalg.norm(P - lm_px[:, m:m + 1, :2], axis=-1), axis=0)
                j = int(np.nanargmin(med)); pick.append(j); errs.append(med[j])
            mapping[side] = tuple(pick)
        e = float(np.nanmedian(errs))
        if best is None or e < best[2]:
            best = (mode, mapping, e)
    return best

def load_rec(subj, cam, ex):
    sd = os.path.join(FIT3D_ROOT, subj)
    c = np.load(cache_path(subj, cam, ex))
    joints = load_joints(os.path.join(sd, 'joints3d_25', ex + '.json'))
    frames = c['frames']; keep = frames < len(joints)
    return dict(lm=c['lm'][keep].astype(float), world=c['world'][keep].astype(float),
                frames=frames[keep], size=tuple(int(v) for v in c['size']), fps=float(c['fps']),
                J=joints[frames[keep]],
                cam=load_camera(os.path.join(sd, 'camera_parameters', cam, ex + '.json')))

print(len(index), 'recordings to check')
if not index:
    raise RuntimeError('Section 1 found no videos. Check FIT3D_ROOT (it must hold s03, s04, ...) '
                       'and that section 1 printed "N videos" with N > 0.')
votes, problems = [], []
for subj, cam, ex in index:
    try:
        r = load_rec(subj, cam, ex)
    except Exception as e:
        problems.append((subj, cam, ex, f'{type(e).__name__}: {e}')); continue
    seen = np.isfinite(r['lm'][:, [11, 12, 13, 14, 23, 24], 0]).all(axis=1)
    if seen.sum() < 20:
        problems.append((subj, cam, ex, f'MediaPipe found shoulders, elbows and hips in only {int(seen.sum())} frames')); continue
    px = r['lm'][seen].copy(); px[..., 0] *= r['size'][0]; px[..., 1] *= r['size'][1]
    res = match(r['J'][seen][::5], px[::5], *r['cam'])
    if res is None:
        problems.append((subj, cam, ex, 'joints project behind the camera in every convention')); continue
    mode, mapping, err = res
    if len(set(mapping['left'] + mapping['right'])) < 6:   # side-on: both hips land on the pelvis
        problems.append((subj, cam, ex, f'two landmarks matched one joint {mapping} - not used for the vote')); continue
    votes.append((subj, cam, ex, mode, str(mapping['left']), str(mapping['right']), round(err, 1)))

if problems:
    print('skipped while matching:')
    for p in problems[:20]:
        print('  ', *p)
V = pd.DataFrame(votes, columns=['subject', 'camera', 'exercise', 'mode', 'left', 'right', 'px_err'])
if V.empty:
    raise RuntimeError('No recording could be matched - send the "skipped while matching" lines above.')
key = V.groupby(['mode', 'left', 'right']).size().sort_values(ascending=False)
print(key.head())
(MODE, LEFT, RIGHT), n_agree = key.index[0], int(key.iloc[0])
MODE, LEFT, RIGHT = int(MODE), tuple(int(v) for v in LEFT.strip('()').split(',')), \
                    tuple(int(v) for v in RIGHT.strip('()').split(','))
JMAP = {'left': LEFT, 'right': RIGHT}
good = V[(V['mode'] == MODE) & (V['left'] == str(LEFT)) & (V['right'] == str(RIGHT))]
print(f'\nusing mode {MODE}, left {LEFT}, right {RIGHT}  ({n_agree}/{len(V)} recordings agree; '
      f'median px error {good.px_err.median():.1f})')
if n_agree < 0.6 * len(V) or good.px_err.median() > 40:
    print('WARNING: weak agreement or large pixel error - send this table before trusting the results')
SH  = {s: JMAP[s][0] for s in JMAP}
EL  = {s: JMAP[s][1] for s in JMAP}
HIP = {s: JMAP[s][2] for s in JMAP}
assert len({*SH.values(), *EL.values(), *HIP.values()}) == 6, f'two landmarks map to one joint: {JMAP}'
print('shoulder', SH, '| elbow', EL, '| hip', HIP)

172 recordings to check
mode  left          right      
0     (11, 12, 1)   (14, 15, 4)    150
                    (14, 15, 0)     21
      (11, 12, 21)  (14, 15, 4)      1
dtype: int64

using mode 0, left (11, 12, 1), right (14, 15, 4)  (150/172 recordings agree; median px error 9.8)
shoulder {'left': 11, 'right': 14} | elbow {'left': 12, 'right': 15} | hip {'left': 1, 'right': 4}


## 3 — Truth and streaming features, per arm

In [6]:
def view_of(J, R, T):
    Xc = to_camera(J, R, T, MODE)
    v = Xc[:, SH['left']] - Xc[:, SH['right']]
    yaw = float(np.nanmedian(np.degrees(np.arctan2(np.abs(v[:, 2]), np.linalg.norm(v[:, :2], axis=1)))))
    return yaw, 'front' if yaw < 30 else 'half' if yaw < 60 else 'profile'

I2D = IDX['angle_2d']
rows, parts, skipped = [], [], {'static': 0, 'occluded': 0, 'align': 0}
t0 = time.time()
for k, (subj, cam, ex) in enumerate(index, 1):
    r = load_rec(subj, cam, ex)
    lm, world, size, fps = r['lm'], r['world'], r['size'], r['fps']
    R, T, f, c = r['cam']
    yaw, view = view_of(r['J'], R, T)
    for side in ('right', 'left'):
        gt = truth_elevation(r['J'], side, SH, EL, HIP)
        rom = float(np.nanpercentile(gt, 95) - np.nanpercentile(gt, 5))
        if rom < MIN_ROM:
            skipped['static'] += 1; continue
        F = stream_shoulder_features(lm, world, side, size, fps, threshold=THRESHOLD)   # frame by frame
        cov = 100.0 * float(np.isfinite(F[:, I2D]).mean())
        if cov < MIN_COV:
            skipped['occluded'] += 1; continue
        lag, lag_r = bc.best_lag(F[:, I2D], gt, max_lag=MAX_LAG, filter_signals=True, fs=fps)
        if abs(lag) >= MAX_LAG or lag_r < 0.7:
            skipped['align'] += 1; continue
        Fa, g = overlap_rows(F, gt, lag)
        ok = np.isfinite(g) & np.isfinite(Fa[:, I2D])
        rec = f'fit3d_{subj}_{cam}_{ex}_{side}'
        parts.append((Fa[ok], g[ok], subj, rec, view, ex))
        rows.append(dict(recording=rec, subject=subj, camera=cam, exercise=ex, side=side, view=view,
                         yaw=round(yaw, 1), lag=lag, r=round(lag_r, 3), frames=int(ok.sum()),
                         raw_mae=round(float(np.mean(np.abs(Fa[ok, I2D] - g[ok]))), 2),
                         elev2d_mae=round(float(np.nanmean(np.abs(Fa[ok, IDX['elev_2d']] - g[ok]))), 2),
                         truth_p5=round(float(np.percentile(g[ok], 5)), 1),
                         truth_p95=round(float(np.percentile(g[ok], 95)), 1)))
    print(f'[{k}/{len(index)}] {subj} {cam} {ex}  {time.time() - t0:.0f}s', flush=True)

REC = pd.DataFrame(rows)
print('kept', len(REC), 'arm-recordings; skipped', skipped)
print('\nRAW BASELINE (median per-recording MAE, deg): angle_2d = what the live system uses today')
REC.groupby('view').agg(recordings=('recording', 'size'), frames=('frames', 'sum'),
                        raw_mae=('raw_mae', 'median'), elev2d_mae=('elev2d_mae', 'median'),
                        truth_p5=('truth_p5', 'median'), truth_p95=('truth_p95', 'median'))

[1/172] s03 50591643 side_lateral_raise  3s
[2/172] s03 58860488 side_lateral_raise  8s
[3/172] s03 60457274 side_lateral_raise  10s
[4/172] s03 65906101 side_lateral_raise  12s
[5/172] s03 50591643 dumbbell_scaptions  14s
[6/172] s03 58860488 dumbbell_scaptions  16s
[7/172] s03 60457274 dumbbell_scaptions  17s
[8/172] s03 65906101 dumbbell_scaptions  19s
[9/172] s03 50591643 dumbbell_overhead_shoulder_press  20s
[10/172] s03 58860488 dumbbell_overhead_shoulder_press  21s
[11/172] s03 60457274 dumbbell_overhead_shoulder_press  23s
[12/172] s03 65906101 dumbbell_overhead_shoulder_press  24s
[13/172] s03 50591643 neutral_overhead_shoulder_press  26s
[14/172] s03 58860488 neutral_overhead_shoulder_press  27s
[15/172] s03 60457274 neutral_overhead_shoulder_press  29s
[16/172] s03 65906101 neutral_overhead_shoulder_press  31s
[17/172] s03 50591643 overhead_trap_raises  33s
[18/172] s03 58860488 overhead_trap_raises  34s
[19/172] s03 60457274 overhead_trap_raises  36s
[20/172] s03 65906101 o

,recordings,frames,raw_mae,elev2d_mae,truth_p5,truth_p95
view,,,,,,
front,311,133916,15.13,14.39,21.8,129.5
half,5,1757,20.76,20.65,22.8,88.6


In [7]:
print(REC.groupby(['exercise', 'view']).agg(recordings=('recording', 'size'), raw_mae=('raw_mae', 'median'),
      truth_p95=('truth_p95', 'median')))

                                        recordings  raw_mae  truth_p95
exercise                         view                                 
dumbbell_overhead_shoulder_press front          63   17.100     135.20
dumbbell_scaptions               front          64   12.020     124.85
neutral_overhead_shoulder_press  front          61   17.860     142.00
overhead_trap_raises             front          46   23.225     149.50
                                 half            1   32.870     143.40
side_lateral_raise               front          60    8.495      98.10
                                 half            4   16.635      88.30
w_raise                          front          17    7.620      71.50


In [8]:
out = os.path.join(OUT, 'feat_fit3d_shoulder.npz')
n = [len(p[1]) for p in parts]
np.savez_compressed(
    out,
    X=np.concatenate([p[0] for p in parts]).astype(np.float32),
    y=np.concatenate([p[1] for p in parts]).astype(np.float32),
    feature_names=np.array(SH_FEATURE_NAMES),
    person=np.repeat([f'fit3d_{p[2]}' for p in parts], n),
    recording=np.repeat([p[3] for p in parts], n),
    view=np.repeat([p[4] for p in parts], n),
    exercise=np.repeat([p[5] for p in parts], n),
    dataset=np.array(['fit3d'] * sum(n)),
)
REC.to_csv(os.path.join(OUT, 'fit3d_shoulder_recordings.csv'), index=False)
print(f'saved {out}: {sum(n):,} frames from {len(parts)} arm-recordings, {len(set(p[2] for p in parts))} people')

saved F:\namal sir project\github repo\AI-based-Physiotherapy-project-\data\fit3d\feat_fit3d_shoulder.npz: 135,673 frames from 316 arm-recordings, 8 people
